# N048 · 链表合并与归并排序

**目标：** 复用节点完成有序合并并分析递归空间。

**先修：** N038, N045, N047。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 有序合并；断链；快慢分段；自底向上归并。

## 从问题到算法

链表不能O(1)随机访问中点，但快慢指针可以线性找到切分点。归并排序递归排序两条更短链，再复用原节点线性合并。相等时优先取左链节点，保留原始次序。本章实现是自顶向下版：递归空间O(log n)，不能声称严格O(1)。

## 最小实现

**本章接口：** `merge_two_lists(a, b)`、`sort_list(head)`

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def list_to_nodes(values):
    dummy = tail = ListNode()
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next

def nodes_to_list(head):
    values, seen = [], set()
    while head is not None:
        if id(head) in seen:
            raise ValueError('cycle in a supposedly finite list')
        seen.add(id(head))
        values.append(head.val)
        head = head.next
    return values
def merge_two_lists(a,b):
    dummy=tail=ListNode()
    while a and b:
        if a.val<=b.val: tail.next=a; a=a.next
        else: tail.next=b; b=b.next
        tail=tail.next
    tail.next=a if a else b
    return dummy.next

def sort_list(head):
    if head is None or head.next is None: return head
    slow=head; fast=head.next
    while fast and fast.next: slow=slow.next; fast=fast.next.next
    second=slow.next; slow.next=None
    return merge_two_lists(sort_list(head),sort_list(second))

## 正确性、前提与复杂度

两个子链长度均严格减小；合并每次取两有序链头较小者，就是全局剩余最小值。链尾仅接剩余链一次，节点集合完整保持。

**代价：** O(n log n)时间、O(log n)递归辅助空间；链表重连不另建数据节点。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[4,2,1,3,2]
show_table(['阶段','链值'],[('输入',a),('排序',nodes_to_list(sort_list(list_to_nodes(a))))])

阶段,链值
输入,"[4, 2, 1, 3, 2]"
排序,"[1, 2, 2, 3, 4]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert nodes_to_list(merge_two_lists(None,list_to_nodes([1,2])))==[1,2]
from random import Random
rng=Random(48)
for n in range(40):
    a=[rng.randrange(5) for _ in range(n)]; h=list_to_nodes(a); before=[]; p=h
    while p: before.append(p); p=p.next
    h=sort_list(h); after=[]; p=h
    while p:
        assert p not in after; after.append(p); p=p.next
    assert [p.val for p in after]==sorted(a)
    assert set(after)==set(before)
    for value in set(a): assert [p for p in after if p.val==value]==[p for p in before if p.val==value]
print('N048: 所有本章断言通过')

N048: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 实现迭代自底向上链表归并。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明递归版并非O(1)栈空间。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 21. Merge Two Sorted Lists（canonical）
- 148. Sort List（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。